In [26]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"
import numpy as np
import torch
import skimage.io as skio
import napari

from tqdm import tqdm
from src.utils.dataset import DatasetSUPPORT_test_stitch, FrameReader,DatasetSUPPORT_incremental_load
from model.SUPPORT import SUPPORT
from src.utils.util import parse_arguments,get_coordinate_generator
import argparse
import random
import logging
import time
from torch.utils.tensorboard import SummaryWriter
from src.utils.dataset import gen_train_dataloader, random_transform
from src.utils.util import parse_arguments
from model.SUPPORT import SUPPORT

viewer = napari.Viewer()


In [ ]:
numFramesToPred = 33
opt = argparse.Namespace(random_seed=0,
                         epoch=10,
                         n_epochs=20,
                         exp_name='mytest2',
                         results_dir=r'C:\Users\WillRemote\Desktop\SUPPORT\SUPPORT\results',
                         input_frames=numFramesToPred,
                         is_folder=False,
                         noisy_data=[r'E:\Synmap Local\4-15-25\Slice 3\FOV 1\acq 4\Analysis\RegBGMov()_1-1.tif'],
                         patch_size=[numFramesToPred, 128, 128],
                         patch_interval=[1, 64, 64],
                         batch_size=8,
                         totalFramesPerEpoch=10000,
                         nConsecFrames=128,
                         model='.\\results\\saved_models\\mytest\\model_6.pth',
                         depth=5,
                         blind_conv_channels=64,
                         one_by_one_channels=[32, 16],
                         last_layer_channels=[64, 32, 16],
                         bs_size=[2, 2],
                         bp=False,
                         unet_channels=[32, 64, 128, 256, 512, 1024],
                         lr=0.0005,
                         loss_coef=[0.5, 0.5],
                         use_CPU=False, n_cpu=8,
                         logging_interval_batch=50,
                         logging_interval=1,
                         sample_interval=10,
                         sample_max_t=600,
                         checkpoint_interval=1)
print(opt)

Namespace(random_seed=0, epoch=0, n_epochs=20, exp_name='mytest2', results_dir='C:\\Users\\WillRemote\\Desktop\\SUPPORT\\SUPPORT\\results', input_frames=33, is_folder=False, noisy_data=['E:\\Synmap Local\\4-15-25\\Slice 3\\FOV 1\\acq 4\\Analysis\\RegBGMov()_1-1.tif'], patch_size=[33, 128, 128], patch_interval=[1, 64, 64], batch_size=8, totalFramesPerEpoch=10000, nConsecFrames=128, model='.\\results\\saved_models\\mytest\\model_6.pth', depth=5, blind_conv_channels=64, one_by_one_channels=[32, 16], last_layer_channels=[64, 32, 16], bs_size=[2, 2], bp=False, unet_channels=[32, 64, 128, 256, 512, 1024], lr=0.0005, loss_coef=[0.5, 0.5], use_CPU=False, n_cpu=8, logging_interval_batch=50, logging_interval=1, sample_interval=10, sample_max_t=600, checkpoint_interval=1)


In [28]:
cuda = torch.cuda.is_available() and (not opt.use_CPU)
cuda

True

In [29]:
random.seed(0)
torch.manual_seed(0)

# ----------
# Initialize: Create sample and checkpoint directories
# ----------
print(opt)

Tensor = torch.cuda.FloatTensor if cuda else torch.Tensor
rng = np.random.default_rng(opt.random_seed)

os.makedirs(opt.results_dir + "/images/{}".format(opt.exp_name), exist_ok=True)
os.makedirs(opt.results_dir + "/saved_models/{}".format(opt.exp_name), exist_ok=True)
os.makedirs(opt.results_dir + "/logs".format(opt.exp_name), exist_ok=True)
logging.basicConfig(level=logging.INFO, filename=opt.results_dir + "/logs/{}.log".format(opt.exp_name),\
    filemode="a", format="%(name)s - %(levelname)s - %(message)s")
writer = SummaryWriter(opt.results_dir + "/tsboard/{}".format(opt.exp_name))

# ----------
# Model, Optimizers, and Loss
# ----------
model = SUPPORT(in_channels=opt.input_frames, mid_channels=opt.unet_channels, depth=opt.depth,\
     blind_conv_channels=opt.blind_conv_channels, one_by_one_channels=opt.one_by_one_channels,\
            last_layer_channels=opt.last_layer_channels, bs_size=opt.bs_size, bp=opt.bp)

optimizer = torch.optim.Adam(model.parameters(), lr=opt.lr)

if cuda:
    model = model.cuda()

# print(opt.results_dir + "/saved_models/%s/model_%d.pth" % (opt.exp_name, opt.epoch-1))
# exit()

if opt.epoch != 0:
    model.load_state_dict(torch.load(opt.results_dir + "/saved_models/%s/model_%d.pth" % (opt.exp_name, opt.epoch-1)))
    optimizer.load_state_dict(torch.load(opt.results_dir + "/saved_models/%s/optimizer_%d.pth" % (opt.exp_name, opt.epoch-1)))
    print('Loaded pre-trained model and optimizer weights of epoch {}'.format(opt.epoch-1))



Namespace(random_seed=0, epoch=0, n_epochs=20, exp_name='mytest2', results_dir='C:\\Users\\WillRemote\\Desktop\\SUPPORT\\SUPPORT\\results', input_frames=33, is_folder=False, noisy_data=['E:\\Synmap Local\\4-15-25\\Slice 3\\FOV 1\\acq 4\\Analysis\\RegBGMov()_1-1.tif'], patch_size=[33, 128, 128], patch_interval=[1, 64, 64], batch_size=8, totalFramesPerEpoch=10000, nConsecFrames=128, model='.\\results\\saved_models\\mytest\\model_6.pth', depth=5, blind_conv_channels=64, one_by_one_channels=[32, 16], last_layer_channels=[64, 32, 16], bs_size=[2, 2], bp=False, unet_channels=[32, 64, 128, 256, 512, 1024], lr=0.0005, loss_coef=[0.5, 0.5], use_CPU=False, n_cpu=8, logging_interval_batch=50, logging_interval=1, sample_interval=10, sample_max_t=600, checkpoint_interval=1)


In [30]:
maxItems = 320000
#approximately 1 hour of training time per batch size of 8
print(opt.epoch)

0


In [ ]:
def train(train_dataloader, model, optimizer, rng, writer, epoch, opt):
    """
    Train a model for a single epoch

    Arguments:
        train_dataloader: (Pytorch DataLoader)
        model: (Pytorch nn.Module)
        optimizer: (Pytorch optimzer)
        rng: numpy random number generator
        writer: (Tensorboard writer)
        epoch: epoch of training (int)
        opt: argparse dictionary

    Returns:
        loss_list: list of total loss of each batch ([float])
        loss_list_l1: list of L1 loss of each batch ([float])
        loss_list_l2: list of L2 loss of each batch ([float])
        corr_list: list of correlation of each batch ([float])
    """

    is_rotate = True if model.bs_size[0] == model.bs_size[1] else False
    
    # initialize
    model.train()
    loss_list_l1 = []
    loss_list_l2 = []
    loss_list = []

    L1_pixelwise = torch.nn.L1Loss()
    L2_pixelwise = torch.nn.MSELoss()

    loss_coef = opt.loss_coef

    # training
    # for i, data in enumerate(tqdm(train_dataloader)):
    #     (noisy_image, _, ds_idx) = data
    #     noisy_image, mean_image, std_image = normalize(noisy_image)
    for i, (noisy_image, b, single_coordinate,mean_image,std_image) in enumerate(tqdm(train_dataloader, desc="train")):
        
        
        noisy_image, _ = random_transform(noisy_image, None, rng, is_rotate)
        

        B, T, X, Y = noisy_image.shape
        noisy_image = noisy_image.cuda()
        noisy_image_target = torch.unsqueeze(noisy_image[:, int(T/2), :, :], dim=1)

        optimizer.zero_grad()
        noisy_image_denoised = model(noisy_image)
        loss_l1_pixelwise = L1_pixelwise(noisy_image_denoised, noisy_image_target)
        loss_l2_pixelwise = L2_pixelwise(noisy_image_denoised, noisy_image_target)
        loss_sum = loss_coef[0] * loss_l1_pixelwise + loss_coef[1] * loss_l2_pixelwise
        loss_sum.backward()
        optimizer.step()

        loss_list_l1.append(loss_l1_pixelwise.item())
        loss_list_l2.append(loss_l2_pixelwise.item())
        loss_list.append(loss_sum.item())

        # print log
        if (epoch % opt.logging_interval == 0) and (i % opt.logging_interval_batch == 0):
            loss_mean = np.mean(np.array(loss_list))
            loss_mean_l1 = np.mean(np.array(loss_list_l1))
            loss_mean_l2 = np.mean(np.array(loss_list_l2))

            ts = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
            writer.add_scalar("Loss_l1/train_batch", loss_mean, epoch*len(train_dataloader) + i)
            writer.add_scalar("Loss_l2/train_batch", loss_mean_l1, epoch*len(train_dataloader) + i)
            writer.add_scalar("Loss/train_batch", loss_mean_l2, epoch*len(train_dataloader) + i)
            
            logging.info(f"[{ts}] Epoch [{epoch}/{opt.n_epochs}] Batch [{i+1}/{len(train_dataloader)}] "+\
                f"loss : {loss_mean:.4f}, loss_l1 : {loss_mean_l1:.4f}, loss_l2 : {loss_mean_l2:.4f}")

    return loss_list, loss_list_l1, loss_list_l2
#aproximately 1 hour of training time

In [101]:
epoch

0

In [99]:
from importlib import reload
from src.utils import dataset

# Reload the modules to ensure any changes are reflected
dataset = reload(dataset)
FrameReader = dataset.FrameReader
DatasetSUPPORT_incremental_load = dataset.DatasetSUPPORT_incremental_load

# ----------
# Training & Validation
# ----------
reader = FrameReader(opt.noisy_data[0])
for epoch in range(opt.epoch, opt.n_epochs):
    #reload random parts of the data every epoch (when too large to fit all in memory)
    train_dataset = DatasetSUPPORT_incremental_load(reader, patch_size=opt.patch_size, patch_interval=opt.patch_interval,maxItems=maxItems)
    dataloader_train = torch.utils.data.DataLoader(train_dataset, batch_size=opt.batch_size)

    loss_list, loss_list_l1, loss_list_l2 =\
        train(dataloader_train, model, optimizer, rng, writer, epoch, opt)

    # logging
    ts = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

    if (epoch % opt.logging_interval == 0):
        loss_mean = np.mean(np.array(loss_list))
        loss_mean_l1 = np.mean(np.array(loss_list_l1))
        loss_mean_l2 = np.mean(np.array(loss_list_l2))

        writer.add_scalar("Loss/train", loss_mean, epoch)
        writer.add_scalar("Loss_l1/train", loss_mean_l1, epoch)
        writer.add_scalar("Loss_l2/train", loss_mean_l2, epoch)
        logging.info(f"[{ts}] Epoch [{epoch}/{opt.n_epochs}] "+\
            f"loss : {loss_mean:.4f}, loss_l1 : {loss_mean_l1:.4f}, loss_l2 : {loss_mean_l2:.4f}")

    if (opt.checkpoint_interval != -1) and (epoch % opt.checkpoint_interval == 0):
        model_loc = opt.results_dir + "/saved_models/%s/model_%d.pth" % (opt.exp_name, epoch)
        torch.save(model.state_dict(), model_loc)
        torch.save(optimizer.state_dict(), opt.results_dir + "/saved_models/%s/optimizer_%d.pth" % (opt.exp_name, epoch))

    # if (epoch % opt.sample_interval == 0):
    #     skio.imsave(opt.results_dir + "/images/%s/denoised_%d.pth" % (opt.exp_name, epoch), )
    

init: torch.Size([132, 889, 1042])


train:   0%|          | 0/40000 [00:00<?, ?it/s]


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [57]:
    
print(opt)
########## Change it with your data ##############
data_file = opt.noisy_data[0]
output_file = "./results/denoised_1.tiff"
patch_size = opt.patch_size
patch_interval = [1, 32, 32]
batch_size = 16    # lower it if memory exceeds.
##################################################

Namespace(random_seed=0, epoch=0, n_epochs=20, exp_name='mytest2', results_dir='C:\\Users\\WillRemote\\Desktop\\SUPPORT\\SUPPORT\\results', input_frames=33, is_folder=False, noisy_data=['E:\\Synmap Local\\4-15-25\\Slice 3\\FOV 1\\acq 4\\Analysis\\RegBGMov()_1-1.tif'], patch_size=[33, 128, 128], patch_interval=[1, 64, 64], batch_size=8, totalFramesPerEpoch=10000, nConsecFrames=128, model='.\\results\\saved_models\\mytest\\model_6.pth', depth=5, blind_conv_channels=64, one_by_one_channels=[32, 16], last_layer_channels=[64, 32, 16], bs_size=[2, 2], bp=False, unet_channels=[32, 64, 128, 256, 512, 1024], lr=0.0005, loss_coef=[0.5, 0.5], use_CPU=False, n_cpu=8, logging_interval_batch=50, logging_interval=1, sample_interval=10, sample_max_t=600, checkpoint_interval=1)


In [58]:
model = SUPPORT(in_channels=opt.input_frames, mid_channels=opt.unet_channels, depth=opt.depth,\
         blind_conv_channels=opt.blind_conv_channels, one_by_one_channels=opt.one_by_one_channels,\
                last_layer_channels=opt.last_layer_channels, bs_size=opt.bs_size, bp=opt.bp).cuda()

In [59]:
epoch

10

In [60]:
epochToLoad = epoch-1
model_loc = opt.results_dir + "/saved_models/%s/model_%d.pth" % (opt.exp_name, epochToLoad)
model_loc

'C:\\Users\\WillRemote\\Desktop\\SUPPORT\\SUPPORT\\results/saved_models/mytest2/model_9.pth'

In [61]:
# model_file
model.load_state_dict(torch.load(model_loc))

<All keys matched successfully>

In [62]:
data_file

'E:\\Synmap Local\\4-15-25\\Slice 3\\FOV 1\\acq 4\\Analysis\\RegBGMov()_1-1.tif'

In [90]:
from importlib import reload
from src.utils import dataset


# Reload the modules to ensure any changes are reflected
dataset = reload(dataset)

DatasetSUPPORT_incremental_load = dataset.DatasetSUPPORT_incremental_load
FrameReader = dataset.FrameReader
reader = FrameReader(data_file,maxFrames=5000,width=588,height=624,gap=1728,shuffle=False)
print(reader.pointer)

0


In [91]:

testset = DatasetSUPPORT_incremental_load(reader, patch_size=patch_size,\
    patch_interval=patch_interval)
testloader = torch.utils.data.DataLoader(testset, batch_size=10)
test_dataloader = testloader

init: torch.Size([132, 889, 1042])


In [92]:
stack_shape = test_dataloader.dataset.output_size
print(stack_shape)
print(reader.pointer)

torch.Size([5000, 889, 1042])
132


In [93]:
print(output_file)
denoised_stack._mmap.close()

./results/denoised_1.tiff


In [94]:
from tifffile import imwrite, memmap

# with h5py.File(output_file, 'w') as hdf5_file:
#     # create an HDF5 dataset to store the denoised stack
#     denoised_stack = hdf5_file.create_dataset("denoised_stack", stack_shape, dtype=np.uint8)
denoised_stack = memmap(output_file, shape=stack_shape, dtype=np.uint16)
with torch.no_grad():
    model.eval()
    # stitching denoised stack
    for _, (noisy_image, _, single_coordinate,mean_image,std_image) in enumerate(tqdm(test_dataloader, desc="validate")):
        
        noisy_image = noisy_image.cuda() #[b, z, y, x]
#             print(noisy_image.shape)
        noisy_image_denoised = model(noisy_image)
        T = noisy_image.size(1)
        for bi in range(noisy_image.size(0)): 
            stack_start_w = int(single_coordinate['stack_start_w'][bi])
            stack_end_w = int(single_coordinate['stack_end_w'][bi])
            patch_start_w = int(single_coordinate['patch_start_w'][bi])
            patch_end_w = int(single_coordinate['patch_end_w'][bi])

            stack_start_h = int(single_coordinate['stack_start_h'][bi])
            stack_end_h = int(single_coordinate['stack_end_h'][bi])
            patch_start_h = int(single_coordinate['patch_start_h'][bi])
            patch_end_h = int(single_coordinate['patch_end_h'][bi])

            stack_start_s = int(single_coordinate['init_s'][bi])

            denoised_stack[stack_start_s+(T//2), stack_start_h:stack_end_h, stack_start_w:stack_end_w] \
                = (noisy_image_denoised[bi].squeeze()[patch_start_h:patch_end_h, patch_start_w:patch_end_w]*std_image[bi] + mean_image[bi]).cpu().numpy()

validate:  24%|██▍       | 89104/372600 [1:06:15<3:30:48, 22.41it/s] 


KeyboardInterrupt: 

In [31]:
import tifffile
with h5py.File(output_file, 'r') as h5_file:
        data = h5_file["denoised_stack"][:]

with tifffile.TiffWriter(output_file+".tif", bigtiff=True) as tiff:
    for i in range(data.shape[0]):
        tiff.save(data[i])


C:\Users\wcunn\AppData\Local\Temp\ipykernel_1576\399134675.py:7: DeprecationWarning: <tifffile.TiffWriter.save> is deprecated. Use TiffWriter.write
  tiff.save(data[i])
